# Notebook 02 — Estimasi Parameter

| Bidang | Detail |
|---|---|
| **Anggota** | Anggota B — Analis Estimasi |
| **Proyek** | `moby/moby` (Docker Engine) |
| **Lapisan** | Minggu 11 — MLE + Beta Posterior |

## Pertanyaan Penelitian yang Dijawab

> **PP1:** *Berapa estimasi probabilitas sebuah Pull Request di `moby/moby` berhasil di-merge, dan bagaimana distribusi Beta posterior menggambarkan ketidakpastian kita terhadap estimasi tersebut?*
>
> **PP2:** *Berapa estimasi rata-rata jumlah isu baru yang dibuka per bulan di `moby/moby`, dan apakah laju tersebut mengikuti proses Poisson?*

---
**Ketergantungan hulu:** Notebook ini menggunakan `data/clean/pull_requests_clean.csv`, `issues_clean.csv`, dan `commits_clean.csv` yang telah disiapkan oleh **Anggota A (Data Engineer)**.

**Ketergantungan hilir:** Estimasi titik (`theta_hat`, `lambda_hat`, `alpha`, `beta`) yang dihasilkan di sini akan digunakan oleh **Anggota C (Analis Inferensi)** di `03_confidence_interval.ipynb`.

## Pengungkapan Penggunaan AI

| # | Tugas | Alat | Ringkasan Prompt | Cara penggunaan output |
|---|------|------|---------------|---------------------|
| 1 | Kerangka awal `estimator.py` | Claude | "Buat fungsi MLE untuk Bernoulli dan Poisson sesuai Tsun 2020" | Ditinjau, rumus α/β dikoreksi menjadi k+1/m+1, struktur dipertahankan |
| 2 | Sel markdown notebook | Claude | "Tulis interpretasi untuk hasil Beta posterior" | Ditulis ulang dengan kata-kata sendiri; pengecekan rumus dilakukan secara manual |

**Ditulis sepenuhnya tanpa AI:** Seluruh paragraf interpretasi statistik, perumusan pertanyaan penelitian, dan kesimpulan ringkasan.

---
## 0. Persiapan Lingkungan

In [ ]:
import sys
import os

# Pastikan src/ ada di path — notebook dijalankan dari folder notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
# Jika dijalankan dari root proyek:
# sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import beta as beta_dist

# Import modul buatan sendiri (Anggota B)
from estimator import (
    mle_bernoulli,
    mle_poisson,
    beta_posterior,
    log_likelihood_bernoulli,
    log_likelihood_poisson,
    plot_log_likelihood_bernoulli,
    plot_log_likelihood_poisson,
    plot_beta_posterior,
)

# Gaya tampilan plot
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('muted')

print('Lingkungan siap ✓')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

---
## 1. Muat & Periksa Data

In [ ]:
# Path — sesuaikan jika dijalankan dari direktori kerja yang berbeda
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data', 'clean')

pr_df     = pd.read_csv(os.path.join(DATA_DIR, 'pull_requests_clean.csv'), parse_dates=['created_at', 'merged_at', 'closed_at'])
issues_df = pd.read_csv(os.path.join(DATA_DIR, 'issues_clean.csv'),        parse_dates=['created_at', 'closed_at'])
commits_df= pd.read_csv(os.path.join(DATA_DIR, 'commits_clean.csv'),       parse_dates=['author_date'])

print(f'PRs     : {len(pr_df):,} baris | kolom: {list(pr_df.columns)}')
print(f'Issues  : {len(issues_df):,} baris | kolom: {list(issues_df.columns)}')
print(f'Commits : {len(commits_df):,} baris | kolom: {list(commits_df.columns)}')

In [ ]:
# Pengecekan cepat pada kolom merge PR
print('Jumlah nilai is_merged pada PR:')
print(pr_df['is_merged'].value_counts())

print('\nJumlah nilai state pada PR:')
print(pr_df['state'].value_counts())

---
## 2. PP1 — MLE Bernoulli: Probabilitas PR Di-Merge

### 2.1 Motivasi

Setiap PR di `moby/moby` memiliki hasil biner: **di-merge** (berhasil) atau **tidak di-merge** (ditutup/terbuka). Kita memodelkan ini sebagai variabel acak Bernoulli:

$$X_i \sim \text{Bernoulli}(\theta), \quad X_i \in \{0, 1\}$$

Tujuannya adalah menemukan nilai θ yang memaksimalkan kemungkinan (*likelihood*) dari data yang kita amati.

### 2.2 Penurunan MLE (Tsun, 2020, hal. 254)

Diberikan n percobaan Bernoulli independen dengan k keberhasilan:

$$L(\theta) = \theta^k (1-\theta)^{n-k}$$

Mengambil log dan menurunkan terhadap θ:

$$\ell(\theta) = k \ln\theta + (n-k)\ln(1-\theta)$$

$$\frac{d\ell}{d\theta} = \frac{k}{\theta} - \frac{n-k}{1-\theta} = 0 \implies \boxed{\hat{\theta} = \frac{k}{n}}$$

In [ ]:
# --- Siapkan kolom hasil biner ---
# Gunakan semua PR yang memiliki hasil pasti (di-merge atau ditutup, bukan masih terbuka)
pr_closed = pr_df[pr_df['state'].isin(['closed'])].copy()
pr_closed['outcome'] = pr_closed['is_merged'].astype(int)

print(f'PR ditutup (hasil pasti): {len(pr_closed):,}')
print(f'  Di-merge              : {pr_closed["outcome"].sum():,}')
print(f'  Ditutup tanpa merge   : {(pr_closed["outcome"] == 0).sum():,}')

In [ ]:
# --- Hitung MLE Bernoulli menggunakan estimator.py ---
bern_result = mle_bernoulli(pr_closed['outcome'].tolist())

print('=== Hasil MLE Bernoulli ===')
print(f"  θ̂ (probabilitas merge) : {bern_result['theta_hat']:.4f}")
print(f"  k (PR di-merge)        : {bern_result['k']:,}")
print(f"  n (total PR ditutup)   : {bern_result['n']:,}")
print(f"  Galat baku SE(θ̂)      : {bern_result['std_error']:.4f}")

### 2.3 Interpretasi

Estimasi MLE memberi tahu kita bahwa sekitar **`theta_hat * 100`%** Pull Request di `moby/moby` berhasil di-merge. Ini adalah nilai θ tunggal yang paling konsisten dengan data yang diamati. Galat baku estimasi yang kecil menunjukkan presisi tinggi mengingat ukuran sampel yang besar.

### 2.4 Kurva Log-Likelihood

In [ ]:
fig = plot_log_likelihood_bernoulli(
    k=bern_result['k'],
    n=bern_result['n'],
    save_path='../report/fig01_ll_bernoulli.png'
)
plt.show()
print('Gambar disimpan → report/fig01_ll_bernoulli.png')

**Interpretasi kurva log-likelihood:**

Kurva memiliki satu maksimum global di θ̂ = k/n, yaitu MLE. Bentuk kurva di sekitar puncak juga memberi gambaran visual tentang ketidakpastian estimator — semakin tajam puncaknya (kelengkungan lebih besar), semakin kecil variansinya. Dengan dataset besar seperti `moby/moby`, puncak kurva sangat curam, mengkonfirmasi galat baku yang rendah.

---
## 3. PP1 (Bayesian) — Beta Posterior untuk Probabilitas Merge

### 3.1 Motivasi

Alih-alih hanya satu titik MLE, kita dapat menghitung **distribusi posterior penuh** atas θ menggunakan prior Beta konjugat. Ini memberi gambaran yang lebih kaya tentang ketidakpastian.

### 3.2 Rumus (Tsun, 2020, hal. 269)

Dimulai dengan **prior seragam** Beta(1, 1), setelah mengamati k keberhasilan dan m kegagalan:

$$p(\theta | k, m) = \text{Beta}(\alpha, \beta), \quad \alpha = k+1, \quad \beta = m+1$$

- **Modus posterior:** $\hat{\theta}_{\text{mode}} = \dfrac{\alpha - 1}{\alpha + \beta - 2}$
- **Rata-rata posterior:** $\hat{\theta}_{\text{mean}} = \dfrac{\alpha}{\alpha + \beta}$

> ⚠️ **Catatan penting:** α = k+1 dan β = m+1, **bukan** k dan m secara langsung. Menggunakan k/m mentah adalah kesalahan rumus (Tsun, 2020, hal. 269).

In [ ]:
k_merges  = bern_result['k']
m_rejects = bern_result['n'] - bern_result['k']   # kegagalan = n - k

bp = beta_posterior(k=k_merges, m=m_rejects)

print('=== Hasil Beta Posterior ===')
print(f"  α (alpha) = k+1         : {bp['alpha']:,}")
print(f"  β (beta)  = m+1         : {bp['beta']:,}")
print(f"  Modus posterior         : {bp['mode']:.4f}")
print(f"  Rata-rata posterior     : {bp['mean']:.4f}")
print(f"  MLE θ̂ (pembanding)     : {bern_result['theta_hat']:.4f}")

print('\nCatatan: modus ≈ MLE θ̂ (keduanya menyatu pada n besar, Tsun hal.269)')

In [ ]:
fig = plot_beta_posterior(
    k=k_merges,
    m=m_rejects,
    save_path='../report/fig02_beta_posterior.png'
)
plt.show()
print('Gambar disimpan → report/fig02_beta_posterior.png')

### 3.3 Interpretasi

Distribusi Beta posterior menunjukkan **rentang penuh nilai θ yang masuk akal**, dibobot berdasarkan bukti dari data kita. Pengamatan utama:

- Distribusi **sangat terkonsentrasi** di dekat MLE, mencerminkan ukuran sampel yang besar (n > 1.000 PR). Dengan begitu banyak data, posterior hampir sepenuhnya didorong oleh *likelihood*, bukan prior.
- **Modus posterior** sama dengan MLE (k/n) hingga koreksi yang dapat diabaikan sebesar 1/(n+2), mengkonfirmasi bahwa estimasi titik frekuentis dan Bayesian setuju pada ukuran sampel ini.
- **Rata-rata posterior** sedikit tertarik ke arah 0,5 dibandingkan modusnya — ini adalah efek *shrinkage* dari prior seragam Beta(1,1).

Beta posterior ini akan diteruskan ke **Anggota C** untuk menghitung **selang kepercayaan Bayesian 95%** (selang ekor sama yang mencakup 95% massa posterior).

---
## 4. PP2 — MLE Poisson: Laju Pembukaan Isu

### 4.1 Motivasi

Jumlah isu baru yang dibuka per bulan di `moby/moby` adalah **proses cacahan** — kandidat alami untuk pemodelan Poisson. Kita ingin mengestimasi laju rata-rata dasar λ (isu/bulan).

### 4.2 Penurunan MLE (Tsun, 2020, hal. 254)

Untuk n observasi Poisson independen $x_1, \ldots, x_n$:

$$L(\lambda) = \prod_{i=1}^{n} \frac{\lambda^{x_i} e^{-\lambda}}{x_i!}$$

Log-likelihood (menghilangkan suku konstanta):

$$\ell(\lambda) = \left(\sum x_i\right)\ln\lambda - n\lambda$$

$$\frac{d\ell}{d\lambda} = \frac{\sum x_i}{\lambda} - n = 0 \implies \boxed{\hat{\lambda} = \frac{\sum x_i}{n} = \bar{x}}$$

In [ ]:
# --- Hitung jumlah isu per bulan ---
issues_df['month'] = issues_df['created_at'].dt.to_period('M')
issues_per_month = issues_df.groupby('month').size().reset_index(name='count')

print('Isu per bulan (sampel):')
print(issues_per_month.tail(10).to_string(index=False))
print(f'\nTotal bulan dengan data: {len(issues_per_month)}')

In [ ]:
# --- Hitung MLE Poisson menggunakan estimator.py ---
count_data = issues_per_month['count'].tolist()
pois_result = mle_poisson(count_data)

print('=== Hasil MLE Poisson ===')
print(f"  λ̂ (rata-rata isu/bulan) : {pois_result['lambda_hat']:.4f}")
print(f"  n (bulan diamati)       : {pois_result['n']}")
print(f"  Galat baku              : {pois_result['std_error']:.4f}")

### 4.3 Visualisasi — Distribusi Jumlah Isu Bulanan

In [ ]:
from scipy.stats import poisson

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Kiri: histogram jumlah bulanan
ax = axes[0]
ax.hist(count_data, bins=15, color='#059669', edgecolor='white', alpha=0.85)
ax.axvline(pois_result['lambda_hat'], color='#dc2626', linestyle='--', linewidth=2,
           label=f"λ̂ = {pois_result['lambda_hat']:.2f}")
ax.set_xlabel('Isu Dibuka per Bulan', fontsize=12)
ax.set_ylabel('Frekuensi', fontsize=12)
ax.set_title('Distribusi Jumlah Isu Bulanan', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.25)

# Kanan: kurva log-likelihood
lambda_hat = pois_result['lambda_hat']
lo = max(0.1, lambda_hat * 0.4)
hi = lambda_hat * 2.0
lam_range = np.linspace(lo, hi, 500)
ll_vals = log_likelihood_poisson(lam_range, np.array(count_data))

ax2 = axes[1]
ax2.plot(lam_range, ll_vals, color='#059669', linewidth=2.5)
ax2.axvline(lambda_hat, color='#dc2626', linestyle='--', linewidth=1.8,
            label=f'MLE λ̂ = {lambda_hat:.2f}')
ax2.scatter([lambda_hat], [log_likelihood_poisson(lambda_hat, np.array(count_data))],
            color='#dc2626', s=80, zorder=5)
ax2.set_xlabel('λ (Laju Poisson)', fontsize=12)
ax2.set_ylabel('ℓ(λ | x)', fontsize=12)
ax2.set_title('Kurva Log-Likelihood Poisson', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.25)

fig.suptitle('moby/moby — Laju Pembukaan Isu (MLE Poisson)', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig('../report/fig03_poisson_issues.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan → report/fig03_poisson_issues.png')

### 4.4 Interpretasi

Estimasi MLE λ̂ mewakili **rata-rata jumlah isu baru yang dibuka per bulan** di `moby/moby`. Kurva log-likelihood mengkonfirmasi bahwa ini adalah maksimum unik — MLE Poisson selalu berupa rata-rata sampel.

Dengan memeriksa histogram, kita dapat mengamati apakah data secara wajar mengikuti bentuk Poisson (kira-kira simetris di sekitar rata-rata untuk λ besar, miring kanan untuk λ kecil). Setiap penyimpangan yang mencolok — seperti bimodalitas atau overdispersi — harus ditandai untuk **Anggota D (Analis Hipotesis)** agar diselidiki dalam uji hipotesis.

---
## 5. Bonus — Laju Aktivitas Commit (MLE Poisson)

In [ ]:
# Commit per minggu sebagai proses Poisson
commits_df['week'] = commits_df['author_date'].dt.to_period('W')
commits_per_week = commits_df.groupby('week').size().reset_index(name='count')

commit_result = mle_poisson(commits_per_week['count'].tolist())

print('=== MLE Poisson — Commit per Minggu ===')
print(f"  λ̂ (rata-rata commit/minggu) : {commit_result['lambda_hat']:.4f}")
print(f"  n (minggu diamati)          : {commit_result['n']}")
print(f"  Galat baku                  : {commit_result['std_error']:.4f}")

---
## 6. Ringkasan Hasil Estimasi

| Parameter | Estimator | Rumus | Nilai | GalBaku |
|---|---|---|---|---|
| θ (prob. merge PR) | MLE Bernoulli | k/n | lihat di atas | lihat di atas |
| α (Beta posterior) | Pembaruan konjugat | k+1 | lihat di atas | — |
| β (Beta posterior) | Pembaruan konjugat | m+1 | lihat di atas | — |
| Modus posterior | Sifat Beta | (α−1)/(α+β−2) | lihat di atas | — |
| Rata-rata posterior | Sifat Beta | α/(α+β) | lihat di atas | — |
| λ (isu/bulan) | MLE Poisson | Σxᵢ/n | lihat di atas | lihat di atas |
| λ (commit/minggu) | MLE Poisson | Σxᵢ/n | lihat di atas | lihat di atas |

In [ ]:
# Ekspor estimasi utama ke CSV untuk Anggota C
import json

estimates = {
    'bernoulli_theta_hat':     bern_result['theta_hat'],
    'bernoulli_k':             bern_result['k'],
    'bernoulli_n':             bern_result['n'],
    'bernoulli_se':            bern_result['std_error'],
    'beta_alpha':              bp['alpha'],
    'beta_beta':               bp['beta'],
    'beta_mode':               bp['mode'],
    'beta_mean':               bp['mean'],
    'poisson_lambda_issues':   pois_result['lambda_hat'],
    'poisson_se_issues':       pois_result['std_error'],
    'poisson_n_issues':        pois_result['n'],
    'poisson_lambda_commits':  commit_result['lambda_hat'],
    'poisson_n_commits':       commit_result['n'],
}

pd.Series(estimates).to_csv('../data/clean/estimation_results.csv', header=['nilai'])

print('Estimasi diekspor → data/clean/estimation_results.csv')
print('\nSemua nilai untuk Anggota C (Analis Inferensi):')
for k, v in estimates.items():
    print(f'  {k:40s}: {v}')

---
## 7. Keterkaitan ke Lapisan Berikutnya

Output berikut dari notebook ini digunakan oleh **Anggota C — Analis Inferensi** di `03_confidence_interval.ipynb`:

| Variabel | Nilai | Tujuan di Lapisan C |
|---|---|---|
| `theta_hat` | MLE Bernoulli | Pusat selang kepercayaan frekuentis untuk laju merge |
| `std_error` | SE(θ̂) | Digunakan dalam rumus SK: θ̂ ± z·SE |
| `n` | Ukuran sampel | Diperlukan untuk rumus selang kepercayaan |
| `alpha`, `beta` | Parameter Beta | Digunakan untuk selang kredibel via `beta_dist.ppf()` |
| `lambda_hat` | MLE Poisson | Pusat selang kepercayaan Poisson |

Semua nilai disimpan di `data/clean/estimation_results.csv`. Anggota C harus mengimpor dari `src/inference.py` dan memuat file ini — **bukan** mengimplementasikan ulang estimator-estimator ini secara langsung.

---
*Akhir Notebook 02 — Lapisan Estimasi*